# Dental Vision V1 — Kaggle GPU diagnostic training
Disk-safe DENTEX diagnostic training. **Use Save Version → Save & Run All (GPU)** so Kaggle runs it server-side even if your Mac sleeps. Training checkpoints every epoch and automatically resumes from an attached prior checkpoint when available.


In [ ]:
!nvidia-smi


In [ ]:
import os, shutil
%cd /kaggle/working
shutil.rmtree('/kaggle/working/dental-vision-v1',ignore_errors=True)
!git clone https://github.com/drhaidarali95/dental-vision-v1.git /kaggle/working/dental-vision-v1
%cd /kaggle/working/dental-vision-v1
!pip -q install -r requirements.txt


In [ ]:
# Remove stale DENTEX data; preserve/copy any prior saved checkpoint.
import pathlib, shutil
root=pathlib.Path('data/dentex')
shutil.rmtree(root,ignore_errors=True)
root.mkdir(parents=True,exist_ok=True)
out=pathlib.Path('/kaggle/working/dentex_diagnostic_v1.pt')
prior=list(pathlib.Path('/kaggle/input').rglob('dentex_diagnostic_v1.pt')) if pathlib.Path('/kaggle/input').exists() else []
if prior:
    shutil.copy2(prior[0],out)
    print('FOUND PRIOR CHECKPOINT:',prior[0],'->',out)
else:
    print('No prior checkpoint attached; starting fresh.')
!python scripts/download_dentex.py --out data/dentex --files training_data.zip
!df -h /kaggle/working


In [ ]:
# Inspect COCO JSONs directly INSIDE the ZIP.
import zipfile,json,pathlib
zpath=root/'training_data.zip'
wanted={'caries','deep caries','periapical lesion','periapical lesions','impacted tooth','impacted teeth'}
candidates=[]
with zipfile.ZipFile(zpath) as z:
    for name in z.namelist():
        if not name.lower().endswith('.json'): continue
        try: d=json.loads(z.read(name))
        except Exception: continue
        if not isinstance(d,dict) or not {'images','annotations','categories'}.issubset(d): continue
        names={str(c.get('name','')).strip().lower() for c in d['categories']}
        score=len(names & wanted)
        print('COCO:',name,'images=',len(d['images']),'annotations=',len(d['annotations']),'categories=',sorted(names),'diagnostic_score=',score)
        if score: candidates.append((score,len(d['images']),name,d))
assert candidates,'No diagnostic COCO JSON found; refusing quadrant fallback.'
score,n,ann_member,d=max(candidates,key=lambda x:(x[0],x[1]))
assert score>=3,f'Only {score} expected diagnostic classes found'
print('SELECTED:',ann_member)
print('CATEGORIES:',[(c.get('id'),c.get('name')) for c in d['categories']])


In [ ]:
# Extract ONLY selected annotation + referenced images, then delete giant ZIP.
import os, shutil
subset=root/'diagnostic_subset'
shutil.rmtree(subset,ignore_errors=True); subset.mkdir(parents=True)
with zipfile.ZipFile(zpath) as z:
    members=z.namelist()
    ann_path=subset/'annotations.json'; ann_path.write_bytes(z.read(ann_member))
    extracted=0
    for im in d['images']:
        fn=str(im['file_name']).replace('\\','/').lstrip('./')
        exact=[m for m in members if m.endswith('/'+fn) or m==fn]
        if not exact: exact=[m for m in members if pathlib.PurePosixPath(m).name==pathlib.PurePosixPath(fn).name]
        if not exact: raise FileNotFoundError(fn)
        src=exact[0]; dest=subset/'images'/pathlib.PurePosixPath(fn).name
        dest.parent.mkdir(parents=True,exist_ok=True)
        with z.open(src) as r, open(dest,'wb') as w: shutil.copyfileobj(r,w)
        im['file_name']=dest.name
        extracted+=1
ann_path.write_text(json.dumps(d))
print('Extracted diagnostic images:',extracted)
zpath.unlink()
print('Deleted archive:',zpath)
!df -h /kaggle/working


In [ ]:
# Train/resume diagnostic detector. train.py writes a checkpoint EVERY epoch.
import subprocess,sys,pathlib
out='/kaggle/working/dentex_diagnostic_v1.pt'
cmd=[sys.executable,'train.py','--images',str(subset/'images'),'--annotations',str(subset/'annotations.json'),'--epochs','20','--batch-size','2','--output',out,'--resume',out]
print('Launching:', ' '.join(cmd))
subprocess.run(cmd,check=True)
p=pathlib.Path(out)
assert p.exists(),'Checkpoint missing after training'
print('CHECKPOINT READY:',p,'MB=',round(p.stat().st_size/1024/1024,1))
print('IMPORTANT: this file is included in the committed Kaggle Version output.')
